In [ ]:
import pandas as pd
import numpy as np

# Reading IT Jobs Data csv to reduce runtime
df = pd.read_csv("SGJobData_IT.csv")

# Pre-Filter: Drop mismatches between positionLevels and employmentTypes

# Identifying invalid pairs of positionLevels and employmentTypes
invalid_pairs = [('Executive', 'Freelance'), ('Executive', 'Internship/Attachment'), ('Junior Executive', 'Freelance'), ('Junior Executive', 'Internship/Attachment'), ('Manager', 'Internship/Attachment'),
                 ('Middle Management', 'Freelance'),  ('Middle Management', 'Internship/Attachment'), ('Middle Management', 'Part Time'), ('Middle Management', 'Temporary'), ('Professional', 'Internship/Attachment'),
                 ('Senior Executive', 'Freelance'), ('Senior Executive', 'Internship/Attachment'), ('Senior Executive', 'Temporary'), ('Senior Management', 'Part Time'), ('Senior Management', 'Temporary')
               ]

# Set reverse_mask to eliminate invalid pairs
reverse_mask = df.set_index(['positionLevels','employmentTypes']).index.isin(invalid_pairs)
mask = ~reverse_mask
df = df[mask]

# Cleaning outliers for salary columns

# Creating new columns salary_min_clean and salary_max_clean
df['salary_min_clean'] = df['salary_minimum']
df['salary_max_clean'] = df['salary_maximum']

# Initial filters, drop rows where salary_maximum < 5
df = df[df['salary_maximum']>=5]

# Fill in outliers for salary_minimum < 1000, set to salary_maximum for cases where only salary_maximum input correctly
# Using criteria "salary_minimum" < 0.2 * "salary_maximum" to filter cases where salary range too wide
df_filter = (df['salary_minimum'] < 1000) &(df['salary_minimum'] < 0.2 * df['salary_maximum'])
df.loc[df_filter,'salary_min_clean'] = df.loc[df_filter,'salary_maximum']

# Wages can be 1. Hourly, 2. Monthly, 3. Annual so we need to clean the salary data accordingly to convert both Hourly and Annual to Monthly salaries

# STEP 1: Handling Hourly Wages

# Clean salary_minimum first by splitting into df_h where possible hourly wage exists
df_h = df[df['salary_min_clean'] < 1000]
df_not_h = df[df['salary_min_clean'] >= 1000]

# Drop rows where senior positions have low salaries
senior_pos = ['Middle Management','Senior Management', 'Senior Executive']
reverse_mask_h = df_h['positionLevels'].isin(senior_pos)
mask_h = ~reverse_mask_h
df_h = df_h[mask_h]

# Filter for hourly wage, by looking per positionLevels and finding low outliers
df_h.groupby("positionLevels")["salary_min_clean"].describe()

# Hourly wages possible when salary_minimum <= 100
hourly_mask = (df_h['salary_min_clean'] <= 100)
df_hour = df_h[hourly_mask]
df_not_hour = df_h[~hourly_mask]

# Convert into monthly wages, converting part-time vs full-time differently
# Using MoM conversion, Monthly Salary = (Hourly Rate × Weekly Hours × 52 weeks) / 12 months
# Full-time average of approx 44.3 hours over time period, Part-time average of 21 hours over time period
# Link: https://stats.mom.gov.sg/Pages/Hours-Worked-Summary-Table.aspx

part_time_mask =(df_hour['employmentTypes'] == "Part Time")

df_hour.loc[part_time_mask, 'salary_min_clean'] = (df_hour.loc[part_time_mask, 'salary_min_clean'] * 21 * 52 / 12).astype(int)
df_hour.loc[part_time_mask, 'salary_max_clean'] = (df_hour.loc[part_time_mask, 'salary_max_clean'] * 21 * 52 / 12).astype(int)
df_hour.loc[~part_time_mask, 'salary_min_clean'] = (df_hour.loc[~part_time_mask, 'salary_min_clean'] * 44.3 * 52 / 12).astype(int)
df_hour.loc[~part_time_mask, 'salary_max_clean'] = (df_hour.loc[~part_time_mask, 'salary_max_clean'] * 44.3 * 52 / 12).astype(int)

# Merge back into df
df_h = pd.merge(df_hour, df_not_hour, how="outer")
df = pd.merge(df_h, df_not_h, how="outer")

# STEP 2: Handling Annual Wages

# Check for "salary_maximum" - Annual vs "salary_minimum" - Monthly
df_max_annual = df[df['salary_min_clean'] < 0.2 * df['salary_max_clean']]
df_max_not_annual = df[df['salary_min_clean'] >= 0.2 * df['salary_max_clean']]

# Set range criteria based on data
sal_high_range = (df_max_annual['salary_max_clean'] > 20500)
sal_low_range = (df_max_annual['salary_max_clean'] <= 20500) 

# For sal_low_range, set both salary_min_clean and salary_max_clean to average_salary to reduce skew
# For sal_high_range, set salary_max_clean to salary_maximum / 12
df_max_annual.loc[sal_low_range,'salary_min_clean'] = df_max_annual.loc[sal_low_range,'average_salary']
df_max_annual.loc[sal_low_range,'salary_max_clean'] = df_max_annual.loc[sal_low_range,'average_salary']
df_max_annual.loc[sal_high_range,'salary_max_clean'] = (df_max_annual.loc[sal_high_range,'salary_max_clean'] / 12).astype(int)

# Merge back into df
df = pd.merge(df_max_annual, df_max_not_annual, how="outer")

# Split into two groups, high vs low wages, remove 1 row of outlier where salary_min and salary_max >2mio
df_high = df[(df['salary_max_clean'] >= 20000) & (df['salary_max_clean'] <= 250000)]
df_not_high = df[df['salary_max_clean'] < 20000]

# Drop rows where junior positions have high monthly/annual salaries
junior_pos = ['Fresh/entry level','Junior Executive']
reverse_mask = df_high['positionLevels'].isin(junior_pos)
mask = ~reverse_mask
df_high = df_high[mask]

# Merge back into df
df = pd.merge(df_high, df_not_high, how="outer")

# Clean both salary_minimum and salary_maximum are Annual salaries, 40,000 used as criteria for both annual salaries
df_both_annual = df[df['salary_min_clean'] > 40000]
df_not_bothannual = df[df['salary_min_clean'] <= 40000]

# Convert Annual to Monthly salaries
df_both_annual['salary_min_clean'] = (df_both_annual["salary_min_clean"] / 12).astype(int)
df_both_annual['salary_max_clean'] = (df_both_annual["salary_max_clean"] / 12).astype(int)

# Merge back into df
df = pd.merge(df_both_annual, df_not_bothannual, how="outer")

# Adding new average_salary_clean column
df['average_salary_clean'] = ((df['salary_min_clean'] + df['salary_max_clean'])/2).astype(int)

# Check if salary_min_clean and salary_max_clean are reasonable
#df["salary_max_clean"].describe()
#df["salary_min_clean"].describe()

# Write to file for testing
#df.to_csv("SGJob_ITSalaryClean.csv",index=False)

